# 📈 Starter Notebook: Predicting Stock Returns

Welcome to the Stock Return Prediction Project! 

The goal of this competition is to predict the forward returns of an asset based strictly on historical price and volume data. 

This notebook provides the bare minimum code to load the data, split it correctly, train a baseline model, and create a submission file. **However, the model in this notebook is intentionally terrible.** It is up to you to engineer the features that will actually find the signal in the noise.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score

plt.style.use('seaborn-v0_8-whitegrid')

# Load the raw data
print("Loading data...")
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

print(f"Training data shape: {train_df.shape}")
print(f"Test data shape: {test_df.shape}")
display(train_df.head())

Loading data...
Training data shape: (1989, 13)
Test data shape: (498, 12)


,id,Date,Open,High,Low,Close,Volume,feature_ret_1d,feature_ret_5d,feature_sma_20_ratio,feature_vol_ratio_20d,feature_high_low_spread,target
0,0,2014-01-30,145.258090,146.054110,144.795091,145.582993,118938100,0.010601,-0.019476,-0.016530,1.018317,0.008648,0.021090
1,1,2014-01-31,143.779734,145.631706,143.706633,144.730087,194677900,-0.005859,-0.003969,-0.021019,1.614905,0.013301,0.032776
2,2,2014-02-03,144.559525,144.884428,141.196732,141.472900,254837100,-0.022505,-0.021572,-0.040753,1.972072,0.026066,0.057817
3,3,2014-02-04,142.106482,142.829401,141.424179,142.463882,165012400,0.007005,-0.020551,-0.032176,1.249410,0.009864,0.043503
4,4,2014-02-05,141.968372,142.601941,141.099250,142.285156,164230500,-0.001255,-0.012292,-0.031169,1.207785,0.010561,0.050979


## 1. Feature Engineering (YOUR TURN)

Machine learning models cannot predict the future just by looking at raw prices. If you feed the model an absolute price like `$450`, it doesn't know if that is relatively high or low. If you use raw prices, your model will overfit and fail on the test set.

You must transform the raw `Open`, `High`, `Low`, `Close`, and `Volume` columns into **relative features** (e.g., percentage changes, moving average ratios, volatility windows).

**Rules for this section:**
1. You can only use past data to create features for today (no looking ahead).
2. You MUST drop the raw price columns before passing the data to your model.

In [2]:
def engineer_features(df):
    """
    Apply all your feature engineering inside this function so you can 
    easily run it on both the train and test sets.
    """
    data = df.copy()
    
    # -------------------------------------------------------------
    # 🛠️ TODO: CREATE YOUR FEATURES HERE
    # -------------------------------------------------------------
    
    # I am giving you ONE basic feature just so the pipeline runs. 
    # This is the 1-day percentage change of the Close price.
    data['feat_daily_return'] = data['Close'].pct_change(1)
    
    # ... create moving averages here ...
    # ... create volume ratios here ...
    # ... create volatility metrics here ...
    
    
    # -------------------------------------------------------------
    # 🧹 CLEANUP: Drop raw absolute prices
    # -------------------------------------------------------------
    columns_to_drop = ['Open', 'High', 'Low', 'Close', 'Volume']
    data = data.drop(columns=columns_to_drop, errors='ignore')
    
    return data

print("Processing features...")
X_train_full = engineer_features(train_df)
X_test = engineer_features(test_df)

# Isolate the target variable
y_train_full = X_train_full['target']

# Isolate just the predictive features
feature_cols = [col for col in X_train_full.columns if col not in ['id', 'target', 'Date']]
X_train_features = X_train_full[feature_cols]
X_test_features = X_test[feature_cols]

print(f"Using {len(feature_cols)} features: {feature_cols}")

Processing features...
Using 6 features: ['feature_ret_1d', 'feature_ret_5d', 'feature_sma_20_ratio', 'feature_vol_ratio_20d', 'feature_high_low_spread', 'feat_daily_return']


## 2. Validation Split

To know if our model actually works, we need to test it on data it hasn't seen during training. Because this is financial time-series data, **we cannot randomly shuffle the rows.** We must split it chronologically.

In [3]:
# Use the first 80% of the training data to train the model
# Use the last 20% to validate its performance
split_index = int(len(X_train_features) * 0.8)

X_train = X_train_features.iloc[:split_index]
y_train = y_train_full.iloc[:split_index]

X_val = X_train_features.iloc[split_index:]
y_val = y_train_full.iloc[split_index:]

print(f"Training on {len(X_train)} rows, Validating on {len(X_val)} rows.")

Training on 1591 rows, Validating on 398 rows.


## 3. Building the Model

When you calculate rolling averages or lagged features, the first few rows of your dataset will contain `NaN` (Not a Number) values. Models crash if they see a `NaN`.

We use a `Pipeline` to automatically fill those missing values (`SimpleImputer`) before passing the data to our `LinearRegression` model.

In [4]:
# Define the pipeline
model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')), 
    ('regressor', LinearRegression())
])

# Train the model
print("Training Linear Regression model...")
model.fit(X_train, y_train)

# Predict on validation fold
val_predictions = model.predict(X_val)

# Evaluate
val_r2 = r2_score(y_val, val_predictions)

print(f"\n✅ Validation R-Squared (R²): {val_r2:.6f}")

if val_r2 <= 0:
    print("⚠️ Your R² is zero or negative. Your model is currently worse than just guessing the average.")
    print("Go back to Step 1 and build better features!")

Training Linear Regression model...

✅ Validation R-Squared (R²): -0.045676
⚠️ Your R² is zero or negative. Your model is currently worse than just guessing the average.
Go back to Step 1 and build better features!


## 4. Generate Final Submission

Once you are happy with your validation score, it is time to generate predictions for `test.csv`. 
We will retrain the model on the *entire* training dataset so it learns from the most recent data, and then predict the future.

In [5]:
# Retrain on the FULL training set
model.fit(X_train_features, y_train_full)

# Predict on the hidden test set
test_predictions = model.predict(X_test_features)

# Format the submission exactly as required
submission = pd.DataFrame({
    'id': test_df['id'],
    'predicted_target': test_predictions
})

# Save to CSV
submission_filename = 'my_submission.csv'
submission.to_csv(submission_filename, index=False)

print(f"🎉 Success! Predictions saved to '{submission_filename}'.")

🎉 Success! Predictions saved to 'my_submission.csv'.
